In [ ]:
import csv
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from learning.bee_colony import main as main_bee
from tqdm import tqdm

# ----------------------------------------------------------------------
# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

# ----------------------------------------------------------------------
# НАСТРОЙКИ ЭКСПЕРИМЕНТА
dataset_name = "mumford2"                     # только mumford2
coefficients = [                              # pp, op, cp
    (0.0, 0.0, 1.0),
    (0.9, 0.1, 0.0)
]
n_routes_options = [26, 36, 46]               # варьируем число маршрутов

model_weights_path = str(
    Path("../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt").resolve()
)

# ----------------------------------------------------------------------
# Директории и CSV-файлы
results_dir = Path("experiment_results/routes_length")
results_dir.mkdir(parents=True, exist_ok=True)

unserved_dir = results_dir / "unserved_demand_matrices"
unserved_dir.mkdir(exist_ok=True)

lc_csv      = results_dir / "lc_results.csv"
neuro_csv   = results_dir / "neurobco_results.csv"

header = [
    "experiment", "n_routes", "pp", "op", "cp",
    "ATT", "RTT", "median_connectivity", "median_connectivity_weighted",
    "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
]

for f in (lc_csv, neuro_csv):
    if not f.exists():
        with open(f, "w", newline="") as fp:
            csv.writer(fp).writerow(header)

metrics_order = [
    'ATT', 'RTT', 'median_connectivity', 'median_connectivity_weighted',
    'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$'
]

# ----------------------------------------------------------------------
# Формируем список экспериментов
experiments = [
    (pp, op, cp, n_routes)
    for pp, op, cp in coefficients
    for n_routes in n_routes_options
]

# ----------------------------------------------------------------------
# Запуск
for pp, op, cp, n_routes in tqdm(experiments,
                                 desc="Running routes_length experiments"):
    param_tag = f"pp{pp:.1f}_op{op:.1f}_cp{cp:.1f}_routes{n_routes}"
    exp_name  = f"routes_length_{dataset_name.upper()}_{param_tag}"

    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp_name}")
    print(f"{'='*70}")

    # ----------------------- LC (initial solution) -----------------------
    try:
        print(f"[1/2] LC → {n_routes} routes")
        lc_run = f"LC_{exp_name}_init"

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_lc = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={lc_run}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={pp}",
                    f"++experiment.cost_function.kwargs.route_time_weight={op}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={cp}",
                    f"++eval.n_routes={n_routes}",
                ]
            )

        lc_metrics, lc_unserved = main_eval(cfg_lc)

        # сохраняем unserved demand
        torch.save(lc_unserved,
                   unserved_dir / f"LC_{exp_name}_unserved_demand.pt")

        # запись в CSV (cost НЕ печатаем в консоль)
        row_lc = [
            exp_name, n_routes, pp, op, cp,
            *[round(lc_metrics.get(k, 0.0).item(), 4)
              if k in lc_metrics else 0.0 for k in metrics_order]
        ]
        with open(lc_csv, "a", newline="") as fp:
            csv.writer(fp).writerow(row_lc)

        print(f"[Success] LC done")

    except Exception as e:
        print(f"[Failed] LC failed: {e}")
        continue

    # ----------------------- NeuroBCO -----------------------
    try:
        print(f"[2/2] NeuroBCO (LC init)")
        neuro_run = f"NEURO_{exp_name}_opt"
        init_path = f"output_routes/nn_construction_{lc_run}_routes.pkl"

        if not Path(init_path).exists():
            print(f"[Warning] init file missing: {init_path}")
            continue

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_neuro = compose(
                config_name="neural_bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={neuro_run}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={pp}",
                    f"++experiment.cost_function.kwargs.route_time_weight={op}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={cp}",
                    f"++eval.n_routes={n_routes}",
                    f"init.path={init_path}",
                ]
            )

        neuro_metrics, neuro_unserved = main_bee(cfg_neuro)
        neuro_metrics['median_connectivity'] /= 60   # как в оригинале

        torch.save(neuro_unserved,
                   unserved_dir / f"NEURO_{exp_name}_unserved_demand.pt")

        row_neuro = [
            exp_name, n_routes, pp, op, cp,
            *[round(neuro_metrics[k].item(), 4) for k in metrics_order]
        ]
        with open(neuro_csv, "a", newline="") as fp:
            csv.writer(fp).writerow(row_neuro)

        print(f"[Success] NeuroBCO done ")

    except Exception as e:
        print(f"[Failed] NeuroBCO failed: {e}")

    print(f"[Completed] {exp_name}\n")

# ----------------------------------------------------------------------
print("\n" + "="*70)
print("=== ALL EXPERIMENTS FINISHED ===")
print("="*70)
print(f"Results: {results_dir}")
print(f"  • LC CSV      : {lc_csv}")
print(f"  • NeuroBCO CSV: {neuro_csv}")
print(f"  • Unserved    : {unserved_dir}/")
print(f"Total runs: {len(experiments)}")